In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import torch
import pickle
import os

# ======================================
# 1️⃣ Load Dataset
# ======================================
path = "../data/raw/cleaned_base_deceptive_reviews.csv"
df = pd.read_csv(path)
print("✅ Loaded:", df.shape)

# Rename columns for consistency
df.rename(columns={'Label': 'label', 'Review': 'review'}, inplace=True)
df = df[['review', 'label']]  # optional: drop others

print("\nColumns after rename:", df.columns.tolist())
display(df.head())


# Optional: preview unique labels
print("\nUnique labels:", df['label'].unique())

# ======================================
# 2️⃣ Clean Text
# ======================================
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)             # remove HTML tags
    text = re.sub(r'http\S+|www.\S+', '', text)   # remove URLs
    text = re.sub(r'[^A-Za-z\s]', '', text)       # remove punctuation/numbers
    text = text.lower().strip()                   # lowercase + trim
    return text

df['cleaned_review'] = df['review'].astype(str).apply(clean_text)

# ======================================
# 3️⃣ Encode Labels
# ======================================
label_map = {'deceptive': 1, 'truthful': 0, 'fake': 1, 'real': 0}
df['label_encoded'] = df['label'].map(label_map)

print("\n✅ After encoding:")
display(df[['review', 'label', 'label_encoded']].head())

# ======================================
# 4️⃣ Split Data
# ======================================
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label_encoded'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_encoded'], random_state=42)

print(f"\n✅ Split complete:")
print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

# ======================================
# 5️⃣ Tokenization (BERT)
# ======================================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(text_list):
    return tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

train_encodings = tokenize_batch(train_df['cleaned_review'].tolist())
val_encodings = tokenize_batch(val_df['cleaned_review'].tolist())
test_encodings = tokenize_batch(test_df['cleaned_review'].tolist())

# ======================================
# 6️⃣ Save Encodings
# ======================================
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/train_encodings.pkl", "wb") as f:
    pickle.dump(train_encodings, f)
with open("data/processed/val_encodings.pkl", "wb") as f:
    pickle.dump(val_encodings, f)
with open("data/processed/test_encodings.pkl", "wb") as f:
    pickle.dump(test_encodings, f)

train_df.to_csv("data/processed/train.csv", index=False)
val_df.to_csv("data/processed/val.csv", index=False)
test_df.to_csv("data/processed/test.csv", index=False)

print("\n✅ All preprocessing done & saved in data/processed/")


✅ Loaded: (1600, 4)

Columns after rename: ['review', 'label']


,review,label
0,We stayed for a one night getaway with family ...,truthful
1,Triple A rate with upgrade to view room was le...,truthful
2,This comes a little late as I'm finally catchi...,truthful
3,The Omni Chicago really delivers on all fronts...,truthful
4,I asked for a high floor away from the elevato...,truthful



Unique labels: ['truthful' 'deceptive']

✅ After encoding:


,review,label,label_encoded
0,We stayed for a one night getaway with family ...,truthful,0
1,Triple A rate with upgrade to view room was le...,truthful,0
2,This comes a little late as I'm finally catchi...,truthful,0
3,The Omni Chicago really delivers on all fronts...,truthful,0
4,I asked for a high floor away from the elevato...,truthful,0



✅ Split complete:
Train: (1120, 4), Val: (240, 4), Test: (240, 4)

✅ All preprocessing done & saved in data/processed/
